# Deploying an ML Model on Azure ML — Customer Churn Prediction
### Cloud Computing | 5BSc DM & DS

Dataset: **IBM Telco Customer Churn** &mdash; 7,043 customers, 19 features (mixed categorical + numeric), binary target `Churn` (Yes/No). This is one of the most widely used datasets in the industry for churn prediction &mdash; telecom, banking, and SaaS companies all build a version of this exact model to flag customers likely to cancel.

Same overall pipeline as before &mdash; **prepare data &rarr; train &rarr; register &rarr; deploy &rarr; test &rarr; clean up** &mdash; but this dataset needs real preprocessing (missing values, categorical encoding), which is a step the earlier all-numeric datasets skipped.

> **Before running:** upload `telco_customer_churn.csv` (provided alongside this notebook) into the same folder as this notebook in Azure ML Studio.

## 0. Prerequisites

1. An **Azure subscription** and an **Azure Machine Learning workspace**
2. A running **Compute Instance** in that workspace
3. This notebook and `telco_customer_churn.csv` uploaded to the same folder in Azure ML Studio
4. Kernel: **Python 3.10 - SDK v2**

## 1. Install and connect the SDK

In [ ]:
%pip install azure-ai-ml azure-identity scikit-learn joblib pandas -q

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Get these three values from: Studio -> top-right workspace name -> download config
subscription_id = "<SUBSCRIPTION_ID>"
resource_group  = "<RESOURCE_GROUP>"
workspace_name  = "<WORKSPACE_NAME>"

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id,
    resource_group,
    workspace_name,
)

print("Connected to workspace:", ml_client.workspace_name)

> **Tip:** inside an Azure ML Studio notebook you can instead use `MLClient.from_config(DefaultAzureCredential())` &mdash; no need to type the three IDs manually.

## 2. Load and explore the data

In [ ]:
import pandas as pd

df = pd.read_csv("telco_customer_churn.csv")

print("Shape:", df.shape)
print(df.dtypes)
df.head()

In [ ]:
# Real corporate data is rarely clean -- inspect for issues before trusting it
print("Missing customerIDs:", df["customerID"].isna().sum())
print("Blank TotalCharges:", (df["TotalCharges"].astype(str).str.strip() == "").sum())
print("Churn distribution:\n", df["Churn"].value_counts(normalize=True))

**Two things worth pointing out to the class:**
- `TotalCharges` is loaded as text, not a number &mdash; a handful of brand-new customers (`tenure == 0`) have it blank.
- The classes are imbalanced (~73% stay, ~27% churn) &mdash; accuracy alone will be a misleading metric later.

## 3. Clean the data

In [ ]:
# TotalCharges should be numeric; blanks correspond to tenure == 0 (brand-new customers)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)

# customerID is an identifier, not a predictive feature
df = df.drop(columns=["customerID"])

# Encode the target: Yes/No -> 1/0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Shape after cleaning:", df.shape)

## 4. Train the model

The categorical columns (contract type, internet service, payment method, etc.) can't go into a model as raw text. Instead of encoding them by hand, we bundle preprocessing **and** the classifier into a single `Pipeline` &mdash; so the exact same transformation used in training is automatically applied at prediction time. This avoids a common production bug called *train-serve skew*.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

X = df.drop("Churn", axis=1)
y = df["Churn"]

categorical_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]
numeric_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numeric_cols),
])

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_train, y_train)

preds = pipeline.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, target_names=["stayed", "churned"]))

joblib.dump(pipeline, "model.pkl")
print("Pipeline (preprocessing + model) saved as model.pkl")

> **Why `class_weight="balanced"`?** With ~73/27 imbalance, a lazy model could get 73% accuracy by always predicting "stayed." This makes the model pay more attention to the minority (churn) class &mdash; usually the one the business actually cares about.

## 5. Register the model in Azure ML

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model_entity = Model(
    path="model.pkl",
    type=AssetTypes.CUSTOM_MODEL,
    name="telco-churn-pipeline-model",
    description="Logistic Regression churn classifier (with preprocessing) on IBM Telco Customer Churn data",
)

registered_model = ml_client.models.create_or_update(model_entity)
print(f"Registered: {registered_model.name}, version {registered_model.version}")

## 6. Write the scoring script

Because `model.pkl` is a full `Pipeline`, the scoring script can hand it **raw** feature values (e.g. `"Contract": "Month-to-month"`) directly &mdash; no manual encoding needed here either.

In [ ]:
%%writefile score.py
import json
import os
import joblib
import pandas as pd

def init():
    """Runs once when the container starts."""
    global model
    model_path = os.path.join(os.getenv("AZUREML_MODEL_DIR"), "model.pkl")
    model = joblib.load(model_path)

def run(raw_data):
    """Runs on every prediction request.
    Expects: {"data": [{"gender": "Female", "tenure": 12, ...}, ...]}
    Returns: list of 0 (stayed) / 1 (churned)
    """
    payload = json.loads(raw_data)
    df = pd.DataFrame(payload["data"])
    predictions = model.predict(df)
    return predictions.tolist()

## 7. Define the environment

In [ ]:
%%writefile conda.yaml
name: telco-churn-env
channels:
  - defaults
dependencies:
  - python=3.10
  - pip
  - pip:
      - scikit-learn
      - joblib
      - azureml-defaults
      - pandas

In [ ]:
from azure.ai.ml.entities import Environment

env = Environment(
    name="telco-churn-env",
    conda_file="conda.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
)

## 8. Create an online endpoint

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import uuid, time

endpoint_name = "churn-endpoint-" + str(uuid.uuid4())[:8]  # must be globally unique

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Endpoint serving the Telco churn classifier",
    auth_mode="key",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# Poll until the endpoint is fully provisioned before deploying to it --
# .result() can return slightly before the endpoint is queryable as "Succeeded",
# which causes an EndpointNotReady error on the next step.
while True:
    state = ml_client.online_endpoints.get(endpoint_name).provisioning_state
    print("Endpoint provisioning state:", state)
    if state == "Succeeded":
        break
    if state == "Failed":
        raise RuntimeError("Endpoint provisioning failed.")
    time.sleep(10)

print("Endpoint ready:", endpoint_name)

## 9. Deploy the model to the endpoint

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=registered_model,
    environment=env,
    code_configuration=CodeConfiguration(
        code=".",             # folder containing score.py
        scoring_script="score.py",
    ),
    instance_type="Standard_DS3_v2",  # Azure's recommended minimum for general-purpose endpoints
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(deployment).result()

# Send 100% of traffic to this deployment
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("Deployment complete.")

> This step takes **5&ndash;10 minutes**. If you see an `EndpointNotReady` error, the polling loop in Step 8 was skipped or interrupted &mdash; re-run Step 8 fully, then this cell.

## 10. Test the deployed endpoint

Send a real customer record from the held-out test set, with its original raw column values (no manual encoding).

In [ ]:
import json

sample = X_test.iloc[[0]].to_dict(orient="records")
true_label = int(y_test.iloc[0])

with open("sample-request.json", "w") as f:
    json.dump({"data": sample}, f)

response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    request_file="sample-request.json",
)
print("Prediction:", response, " (0=stayed, 1=churned)")
print("Actual label:", true_label)

In [ ]:
endpoint_details = ml_client.online_endpoints.get(endpoint_name)
keys = ml_client.online_endpoints.get_keys(endpoint_name)

print("Scoring URI:", endpoint_details.scoring_uri)
print("Primary key:", keys.primary_key)

## 11. Clean up (avoids ongoing charges)

In [ ]:
ml_client.online_endpoints.begin_delete(name=endpoint_name).result()
print("Endpoint deleted.")

Also **stop the compute instance** from the Studio UI when finished.

## Talking points / possible viva questions

- Why does bundling preprocessing into the `Pipeline` (Step 4) matter for deployment, compared to encoding the data by hand before training?
- Why is `class_weight="balanced"` relevant here but wasn't needed for the Iris or Breast Cancer demos?
- What real business action would a telecom company take with this model's output? (Hint: think retention offers, not just a number.)
- Why did we drop `customerID` before training? What would happen if we accidentally left it in?
- Accuracy alone said the model looked decent &mdash; why does the classification report matter more here?

**Dataset source:** IBM Telco Customer Churn sample dataset (public domain sample data, mirrored via GitHub for this exercise).